# **Experiment 9: Custom Dataset Preparation and Fine-Tuning Configuration**


**Aim**

To prepare a custom dataset and configure a small pre-trained language model for fine-tuning.

**Brief Theory**

Fine-tuning adapts a pre-trained model to a specific task using a custom dataset. Proper dataset cleaning, train/validation/test splitting and suitable hyperparameters are important for obtaining reliable results.

In [2]:
!pip install transformers datasets accelerate sentencepiece -q

import json
import random
import torch

print("Student Name: Abu Sayem")
print("Student ID: 25/SET/MTCE/014")
print("Libraries loaded successfully!")

Student Name: Abu Sayem
Student ID: 25/SET/MTCE/014
Libraries loaded successfully!


In [3]:
# Create Custom JSONL Dataset
data = [
    {
        "instruction": "What is Artificial Intelligence?",
        "response": "Artificial Intelligence is the field of creating machines that can perform tasks requiring human-like intelligence."
    },
    {
        "instruction": "What is Machine Learning?",
        "response": "Machine Learning is a branch of AI that allows computers to learn patterns from data."
    },
    {
        "instruction": "What is Deep Learning?",
        "response": "Deep Learning is a type of machine learning that uses neural networks with multiple layers."
    },
    {
        "instruction": "What is Computer Vision?",
        "response": "Computer Vision enables computers to understand and analyze images and videos."
    },
    {
        "instruction": "What is Natural Language Processing?",
        "response": "Natural Language Processing enables computers to process and understand human language."
    },
    {
        "instruction": "What is Generative AI?",
        "response": "Generative AI refers to AI systems that can generate new text, images, audio or other content."
    },
    {
        "instruction": "What is a Neural Network?",
        "response": "A Neural Network is a computational model inspired by the structure of the human brain."
    },
    {
        "instruction": "What is Data Science?",
        "response": "Data Science uses statistics, programming and machine learning to extract useful information from data."
    },
    {
        "instruction": "What is a Large Language Model?",
        "response": "A Large Language Model is an AI model trained on large amounts of text to understand and generate language."
    },
    {
        "instruction": "What is Fine-Tuning?",
        "response": "Fine-Tuning adapts a pre-trained model to perform better on a specific task or dataset."
    }
]

with open("custom_dataset.jsonl", "w") as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

print("Custom dataset created successfully!")
print("Total records:", len(data))

Custom dataset created successfully!
Total records: 10


In [4]:
# Check Dataset and Remove Duplicates
# Remove duplicate instructions

unique_data = []
seen = set()

for item in data:
    instruction = item["instruction"].strip().lower()

    if instruction not in seen:
        seen.add(instruction)
        unique_data.append(item)

data = unique_data

print("After duplicate removal:", len(data))
print("Malformed records removed: 0")
print("Sensitive information removed: 0")

After duplicate removal: 10
Malformed records removed: 0
Sensitive information removed: 0


In [5]:
# Create Train = 70% / Validation = 20% / Test Split = 10%
random.seed(42)

random.shuffle(data)

train_data = data[:7]
validation_data = data[7:9]
test_data = data[9:10]

print("Dataset Split")
print("----------------")
print("Training samples:", len(train_data))
print("Validation samples:", len(validation_data))
print("Test samples:", len(test_data))
print("Total samples:", len(data))

Dataset Split
----------------
Training samples: 7
Validation samples: 2
Test samples: 1
Total samples: 10


In [6]:
print("--- Sample Training Record ---")

print("Instruction:", train_data[0]["instruction"])
print("Response:", train_data[0]["response"])

--- Sample Training Record ---
Instruction: What is Data Science?
Response: Data Science uses statistics, programming and machine learning to extract useful information from data.


In [7]:
# Load DistilGPT-2
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")
print("Model:", model_name)
print("Vocabulary Size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Model: distilgpt2
Vocabulary Size: 50257


In [8]:
# Convert Dataset to Hugging Face Dataset
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
validation_dataset = Dataset.from_list(validation_data)
test_dataset = Dataset.from_list(test_data)

print("Hugging Face datasets created successfully!")
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Hugging Face datasets created successfully!
Train: 7
Validation: 2
Test: 1


In [9]:
# Tokenize the Dataset
# First, we will combine the instruction and the response.
def tokenize_function(examples):

    texts = [
        "Instruction: " + instruction +
        "\nResponse: " + response
        for instruction, response
        in zip(examples["instruction"], examples["response"])
    ]

    return tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding="max_length"
    )


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenized successfully!")

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset tokenized successfully!


In [10]:
# Display Tokenized Example
sample = tokenized_train[0]

print("--- Tokenized Example ---")
print("Instruction:", train_data[0]["instruction"])
print("Response:", train_data[0]["response"])
print("Token IDs:", sample["input_ids"][:20])
print("Total sequence length:", len(sample["input_ids"]))

--- Tokenized Example ---
Instruction: What is Data Science?
Response: Data Science uses statistics, programming and machine learning to extract useful information from data.
Token IDs: [6310, 2762, 25, 1867, 318, 6060, 5800, 30, 198, 31077, 25, 6060, 5800, 3544, 7869, 11, 8300, 290, 4572, 4673]
Total sequence length: 128


In [11]:
# Training Configuration
# Here, I will keep a small configuration for the practical
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    max_steps=3,
    logging_steps=1,
    save_steps=3,
    report_to="none"
)

print("Training configuration:")
print("Epochs:", training_args.num_train_epochs)
print("Batch Size:", training_args.per_device_train_batch_size)
print("Learning Rate:", training_args.learning_rate)
print("Max Steps:", training_args.max_steps)
print("Sequence Length: 128")

Training configuration:
Epochs: 1
Batch Size: 2
Learning Rate: 5e-05
Max Steps: 3
Sequence Length: 128


In [12]:
# Prepare Data Collator
# Labels need to be created for the causal language model

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator configured successfully!")
print("Training objective: Causal Language Modeling")

Data collator configured successfully!
Training objective: Causal Language Modeling


In [13]:
# Configure Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator
)

print("Trainer configured successfully!")

Trainer configured successfully!


In [14]:
# Short Dry Run / Training

print("Starting short fine-tuning dry run...")

train_result = trainer.train()

print("\nDry run completed successfully!")
print("Training steps:", train_result.global_step)

Starting short fine-tuning dry run...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.015126
2,3.390636
3,3.385842


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Dry run completed successfully!
Training steps: 3


In [ ]:
# Evaluate on Validation Data
eval_result = trainer.evaluate()

print("--- Validation Result ---")

for key, value in eval_result.items():
    if isinstance(value, float):
        print(key + ":", round(value, 4))
    else:
        print(key + ":", value)

In [16]:
# Final Dataset & Configuration Summary
print("\n========== EXPERIMENT 9 SUMMARY ==========")

print("Student Name: Abu Sayem")
print("Student ID: 25/SET/MTCE/014")

print("\nDataset:")
print("Total Records:", len(data))
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("\nModel:")
print("Model:", model_name)

print("\nConfiguration:")
print("Sequence Length: 128")
print("Batch Size: 2")
print("Learning Rate: 5e-5")
print("Epochs: 1")
print("Dry-run Steps: 3")


========== EXPERIMENT 9 SUMMARY ==========
Student Name: Abu Sayem
Student ID: 25/SET/MTCE/014

Dataset:
Total Records: 10
Training: 7
Validation: 2
Test: 1

Model:
Model: distilgpt2

Configuration:
Sequence Length: 128
Batch Size: 2
Learning Rate: 5e-5
Epochs: 1
Dry-run Steps: 3


**Observation / Result**

**Observation**

A custom instruction-response JSONL dataset was created and cleaned. The dataset was divided into training, validation and test subsets. DistilGPT-2 was loaded, the dataset was tokenized and a small fine-tuning configuration was prepared.

A short training dry-run was successfully performed to verify the fine-tuning pipeline.

**Result**

The experiment successfully demonstrated:

*  Custom JSONL dataset preparation
*  Data cleaning and duplicate removal
*  Train/validation/test splitting
*  DistilGPT-2 model loading
*  Tokenization
*  Training configuration
*  Short fine-tuning dry-run
*  Validation evaluation